<a href="https://colab.research.google.com/github/Wizard-i/FlyRank-Starter-Repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wizard-i/FlyRank-Starter-Repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)


### My lane as an ML task: Scoring / Ranking

I choose a supervised scoring and ranking approach for the Refresh / Content Opportunity Scoring lane.

The goal is to assign a review-priority score to each content page and rank the pages from highest to lowest priority.

For the starter stage, a binary proxy label will help estimate the likelihood that a page belongs to the currently declining group. The resulting score can support the prioritization of pages for human review.

The final ranked output will help content editors decide which pages need further investigation, refreshing, expansion, protection, or monitoring.

The main objective is not simply to predict a label, but to help editors decide which pages deserve attention first.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

# Load the public anonymized starter dataset
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Wizard-i/FlyRank-Starter-Repo/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df_raw = pd.read_csv(DATA_URL)

# Required columns for the selected lane
required_columns = [
    "content_id",
    "client_id",
    "content_age_days",
    "impressions_90d",
    "trend_direction"
]

missing_columns = [
    col for col in required_columns
    if col not in df_raw.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Convert numeric fields safely
for col in ["content_age_days", "impressions_90d"]:
    df_raw[col] = pd.to_numeric(
        df_raw[col], errors="coerce"
    )

# Prepare the starter lane slice
df_lane = df_raw.loc[
    (df_raw["impressions_90d"] > 0) &
    (df_raw["content_age_days"] >= 90)
].copy()

# Keep one row per content page
df_lane = (
    df_lane
    .drop_duplicates(subset=["content_id"])
    .reset_index(drop=True)
)

print("Lane data loaded successfully!")
print("Raw dataset rows:", len(df_raw))
print("Prepared lane rows:", len(df_lane))
print("Unique content pages:", df_lane["content_id"].nunique())


Lane data loaded successfully!
Raw dataset rows: 30000
Prepared lane rows: 30000
Unique content pages: 30000


## 2. Target or proxy


My initial target proxy is `is_declining_proxy`.

It is a binary target:

- 1 = The page has a currently declining trend.
- 0 = The page does not have a currently declining trend.

The proxy is derived from the observed `trend_direction` field in the starter dataset.

This is a beginner-level proxy, not a true future-outcome label. It does not indicate whether refreshing a page will improve its performance.

The model will estimate the likelihood of this proxy, and the resulting probability can be used as an initial review-priority score.

For future capstone work, I would prefer a stronger target based on prior 90-day signals and an independently observed outcome in the following 30 days.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



if df_lane["trend_direction"].isna().any():
    raise ValueError("Missing trend_direction values found.")

df_lane["is_declining_proxy"] = (
    df_lane["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype("int8")
)

print("Target proxy created successfully!")

print("\nTarget distribution:")
print(
    df_lane["is_declining_proxy"]
    .value_counts()
    .sort_index()
)

print("\nTarget percentage:")
print(
    (
        df_lane["is_declining_proxy"]
        .value_counts(normalize=True)
        .sort_index() * 100
    ).round(2)
)

print("\nTarget examples:")
display(
    df_lane[
        ["content_id", "is_declining_proxy"]
    ].head(10)
)


Target proxy created successfully!

Target distribution:
is_declining_proxy
0    13738
1    16262
Name: count, dtype: int64

Target percentage:
is_declining_proxy
0    45.79
1    54.21
Name: proportion, dtype: float64

Target examples:


,content_id,is_declining_proxy
0,content_304f48230142,1
1,content_a1fb4e703a9e,1
2,content_9aa793d4d895,1
3,content_331d6c4de07b,0
4,content_d99b7a2d90ca,1
5,content_d4084a4bc775,1
6,content_9a34b442b552,1
7,content_a63219c6e95a,0
8,content_5e6c160719bc,1
9,content_c27558df2b0c,1


## 3. Success metric

My primary evaluation metric will be Precision@20.

It measures the proportion of relevant pages among the top 20 pages ranked by the model's predicted scores.

I selected this metric because content editors have limited time and need a useful shortlist of pages to review first.

A higher Precision@20 means that more of the top-ranked pages match the selected declining-proxy label.

During model evaluation, I will compare the result against a transparent rule-based baseline and the overall proxy-positive rate.

The metric will evaluate how well the system ranks pages according to the starter proxy. It will not prove that refreshing those pages will improve their search performance.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


REVIEW_K = 20

def precision_at_k(y_true, scores, k=REVIEW_K):

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    if len(y_true) != len(scores):
        raise ValueError(
            "Target and score arrays must have equal length."
        )

    if len(y_true) == 0:
        raise ValueError("No observations available.")

    if k <= 0:
        raise ValueError("K must be positive.")

    k = min(k, len(y_true))

    top_indices = np.argsort(scores)[::-1][:k]

    return float(y_true[top_indices].mean())


print("Primary metric: Precision@20")
print("Review batch size:", REVIEW_K)
print(
    "Formula: Relevant pages in the top 20 "
    "/ 20 reviewed pages"
)


Primary metric: Precision@20
Review batch size: 20
Formula: Relevant pages in the top 20 / 20 reviewed pages


## 4. The unit of analysis, as a real dataframe

The unit of analysis is one pseudonymized content page.

Each row represents one unique content item, identified by a pseudonymous `content_id`.

The dataset provides content properties and aggregated search and engagement measurements over a trailing 90-day window.

For this lane, I use pages with at least one search impression and a content age of at least 90 days, following the starter preparation process.

The `content_id` is used only to identify and distinguish pages, not as a model feature.

The target proxy is included as a separate column so that the structure of the intended learning problem is visible.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

display_columns = [
    "content_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "is_declining_proxy"
]

# Check that all display columns are available
missing_display_columns = [
    col for col in display_columns
    if col not in df_lane.columns
]

if missing_display_columns:
    raise ValueError(
        f"Missing display columns: {missing_display_columns}"
    )

# Display actual page-level observations
page_dataframe = df_lane[display_columns].copy()

print("UNIT OF ANALYSIS: ONE ROW = ONE CONTENT PAGE")
print("DataFrame shape:", page_dataframe.shape)
print("Unique pages:", page_dataframe["content_id"].nunique())
print(
    "Duplicate page IDs:",
    page_dataframe["content_id"].duplicated().sum()
)

display(page_dataframe.head(10))


UNIT OF ANALYSIS: ONE ROW = ONE CONTENT PAGE
DataFrame shape: (30000, 11)
Unique pages: 30000
Duplicate page IDs: 0


,content_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,is_declining_proxy
0,content_304f48230142,keyword article,187,20,3803,29,0.76,10.6,17,5.88,1
1,content_a1fb4e703a9e,keyword article,445,25,15320,7,0.05,20.3,9,0.00,1
2,content_9aa793d4d895,keyword article,141,20,12581,11,0.09,36.5,11,0.00,1
3,content_331d6c4de07b,keyword article,463,22,11751,58,0.49,6.2,78,1.28,0
4,content_d99b7a2d90ca,keyword article,263,14,19140,24,0.13,44.0,145,0.00,1
5,content_d4084a4bc775,keyword article,147,20,3970,1,0.03,8.5,5,0.00,1
6,content_9a34b442b552,keyword article,90,20,20,0,0.00,7.0,1,0.00,1
7,content_a63219c6e95a,keyword article,445,22,1724,1,0.06,21.2,28,3.57,0
8,content_5e6c160719bc,keyword article,90,20,32574,29,0.09,46.0,68,5.88,1
9,content_c27558df2b0c,keyword article,257,104,1240,2,0.16,4.9,3,0.00,1


## 5. Why ML beats a fixed rule here

A fixed rule can identify simple situations, such as an old page with high search impressions or a visible page with a low CTR.

However, content opportunities may involve several interacting signals, including freshness, search visibility, engagement, content depth, and search intent.

A machine learning approach may learn useful combinations of these signals and assign a more flexible review-priority score than a small set of fixed thresholds.

I will not assume that ML is automatically better. I will first establish a transparent rule-based baseline and compare it with the ML approach using Precision@20.

If the model does not provide a meaningful improvement, the simpler and more explainable rule-based approach may be the better choice.

The final output will support human review decisions. It will not automatically determine that a page must be refreshed or that a refresh will improve performance.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

candidate_features = [
    "search_volume",
    "competition",
    "word_count",
    "content_type",
    "main_intent",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate"
]

# Keep only columns available in the dataset
candidate_features = [
    col for col in candidate_features
    if col in df_lane.columns
]

# Fields that must not become model features
excluded_fields = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
    "is_declining_proxy",
    "provider_used",
    "model_used"
}

# Check feature selection
leakage_found = (
    set(candidate_features) & excluded_fields
)

assert not leakage_found, (
    f"Potential leakage or identifier fields: {leakage_found}"
)

print("Candidate feature count:", len(candidate_features))
print("\nCandidate features:")
print(candidate_features)

print("\nExcluded fields:")
print(sorted(excluded_fields))

print("\nLeakage check passed!")


Candidate feature count: 14

Candidate features:
['search_volume', 'competition', 'word_count', 'content_type', 'main_intent', 'content_age_days', 'days_since_last_update', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']

Excluded fields:
['client_id', 'content_id', 'impressions_last_30d', 'impressions_prev_30d', 'is_declining_proxy', 'model_used', 'provider_used', 'trend_direction', 'trend_pct']

Leakage check passed!


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.